# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAhmadIshtiaq/ml-internship-muhammadahmadishtiaq/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding — "What Predicts Health?" (ML Appendix, Feature Importance, p.27).** A Random Forest finds `avg_position` (43%), `impressions` (32%), and `scroll_depth` (15%) as the top predictors of `health_score`.

*My methodology question:* where does the label come from? The paper itself states `health_score = impressions(30) + position(30) + CTR(20) + scroll_depth(20)` — so `avg_position` and `impressions` aren't independent predictors of the target, they're literally two of the four terms that sum to build it. This is the same structural leak I flagged in my own ML-05 audit (features that are one arithmetic step from the label). To the paper's credit, it says this outright — "the target itself is partly constructed from some of these inputs, so importance is descriptive rather than causal" — which is exactly the right caveat. The constructive ask would just be to state it a beat earlier, before the ranked bar chart, so a skimming reader doesn't read "43% importance" as a discovery rather than an expected consequence of the formula.

**Finding — "What Predicts Growth?" (ML Appendix, Growth & Classification, p.29).** A logistic regression reports 71% holdout accuracy separating growing from declining pages, using `content_age`, `days_since_update`, and `days_visible` as the strongest signals.

*My methodology question:* does the validation design carry the claim? The label here — growth vs. decline from 30d-vs-prev-30d impression change — is the exact same trend-based proxy my own capstone lane uses, so I trust the label source. What isn't stated is *how* the 80/20 holdout was drawn: portfolio pages come from 57 brands, and pages from the same brand likely share template, domain authority, and publishing cadence. If the split was random across all pages rather than grouped by brand, the reported 71% could be partly the model recognizing "this looks like a Brand X page" rather than a transferable content signal — exactly the gap I found in my own ML-08 model, where a grouped-by-client split moved the numbers versus a naive split. Since it's the same lane and same label type as my own work, this is the one methodology question worth flagging most: was the 80/20 split grouped by brand, and if not, would a grouped re-run change the reported 71%?

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# No computation needed for this section -- confirming the two cited numbers straight from the paper's own text,
# so the critique above is checked against the source rather than paraphrased from memory.
paper_health_score_formula = "impressions(30) + avg_position(30) + CTR(20) + scroll_depth(20)"
paper_top3_importance = {"avg_position": 0.43, "impressions": 0.32, "scroll_depth": 0.15}
paper_growth_model_accuracy = 0.71

overlap_with_formula = set(paper_top3_importance) & {"avg_position", "impressions", "scroll_depth"}
print("Health-score formula components:", paper_health_score_formula)
print("Top-3 RF importance features:", paper_top3_importance)
print("Overlap between top predictors and the label's own formula:", overlap_with_formula)
print(f"\nReported growth-model holdout accuracy: {paper_growth_model_accuracy:.0%} (split-grouping not stated in the paper)")


Health-score formula components: impressions(30) + avg_position(30) + CTR(20) + scroll_depth(20)
Top-3 RF importance features: {'avg_position': 0.43, 'impressions': 0.32, 'scroll_depth': 0.15}
Overlap between top predictors and the label's own formula: {'avg_position', 'impressions', 'scroll_depth'}

Reported growth-model holdout accuracy: 71% (split-grouping not stated in the paper)


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-running the exact ML-08 Logistic Regression, once under a **naive random split** (the "before" — what my numbers would look like if I hadn't grouped by client) and once under the **grouped-by-client split** already used in ML-08 (the "after"). Same features, same model, same metric — only the split changes.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# --- same feature vector as ML-05/ML-08 ---
feat = pd.DataFrame(index=df.index)
feat["has_word_count"] = df["word_count"].notna().astype(int)
feat["has_keyword_data"] = df["search_volume"].notna().astype(int)
feat["has_clicks"] = (df["clicks_90d"] > 0).astype(int)
feat["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
numeric_cols = [
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "search_volume", "competition", "cpc",
]
for col in numeric_cols:
    feat[col] = df[col].fillna(0)
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    feat[f"log_{col}"] = np.log1p(feat[col])
categorical_cols = [
    "content_type", "main_intent", "competition_level",
    "age_tier", "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier",
]
cat_df = df[categorical_cols].fillna("unknown")
feat = pd.concat([feat, pd.get_dummies(cat_df, prefix=categorical_cols)], axis=1)

X = feat
y = df["is_declining_label"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def eval_split(train_idx, test_idx, label):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42))
    model.fit(X_tr, y_tr)
    scores = model.predict_proba(X_te)[:, 1]
    row = {
        "precision@10": precision_at_k(scores, y_te, 10),
        "precision@20": precision_at_k(scores, y_te, 20),
        "precision@50": precision_at_k(scores, y_te, 50),
        "roc_auc": roc_auc_score(y_te, scores),
        "n_test": len(test_idx),
    }
    print(f"{label}: {row}")
    return row

# BEFORE: naive random 75/25 split, no grouping (client_id ignored)
train_idx_naive, test_idx_naive = train_test_split(
    np.arange(len(df)), test_size=0.25, random_state=42, stratify=y
)
before = eval_split(train_idx_naive, test_idx_naive, "BEFORE (naive random split)")

# AFTER: grouped by client_id, same as ML-08
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx_grp, test_idx_grp = next(gss.split(df, y, groups=df["client_id"]))
after = eval_split(train_idx_grp, test_idx_grp, "AFTER (grouped-by-client split)")

before_after = pd.DataFrame({"before_naive_split": before, "after_grouped_split": after}).T
print("\nBefore vs after comparison:")
print(before_after.round(3))


BEFORE (naive random split): {'precision@10': np.float64(1.0), 'precision@20': np.float64(0.9), 'precision@50': np.float64(0.9), 'roc_auc': 0.7405943089998586, 'n_test': 7500}


AFTER (grouped-by-client split): {'precision@10': np.float64(1.0), 'precision@20': np.float64(0.8), 'precision@50': np.float64(0.74), 'roc_auc': 0.6352605600379686, 'n_test': 7115}

Before vs after comparison:
                     precision@10  precision@20  precision@50  roc_auc  n_test
before_naive_split            1.0           0.9          0.90    0.741  7500.0
after_grouped_split           1.0           0.8          0.74    0.635  7115.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same three-part hunt as ML-05, re-run on this notebook's final feature set: label-derived column test, in-window sibling-metric check, and a scan for existing product flags.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

forbidden = {"trend_pct", "trend_direction", "impressions_last_30d", "impressions_prev_30d"}
in_final_features = set(X.columns) & forbidden
print("Forbidden columns present in final feature set:", in_final_features, "(empty = clean)")

# Re-confirm the label-derived leak test from ML-05, on the grouped split used above
X_grp_test = X.iloc[test_idx_grp].copy()
X_grp_train = X.iloc[train_idx_grp].copy()
y_grp_train, y_grp_test = y.iloc[train_idx_grp], y.iloc[test_idx_grp]

X_with_leak_train = X_grp_train.copy()
X_with_leak_train["impressions_last_30d"] = df.iloc[train_idx_grp]["impressions_last_30d"].fillna(0)
X_with_leak_train["impressions_prev_30d"] = df.iloc[train_idx_grp]["impressions_prev_30d"].fillna(0)
X_with_leak_test = X_grp_test.copy()
X_with_leak_test["impressions_last_30d"] = df.iloc[test_idx_grp]["impressions_last_30d"].fillna(0)
X_with_leak_test["impressions_prev_30d"] = df.iloc[test_idx_grp]["impressions_prev_30d"].fillna(0)

leak_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=42))
leak_model.fit(X_with_leak_train, y_grp_train)
leak_scores = leak_model.predict_proba(X_with_leak_test)[:, 1]
leak_auc = roc_auc_score(y_grp_test, leak_scores)
honest_auc = after["roc_auc"]
print(f"\nHonest ROC-AUC (final feature set): {honest_auc:.3f}")
print(f"ROC-AUC with suspect columns added back: {leak_auc:.3f}")
print(f"Gap: {leak_auc - honest_auc:+.3f} -- confirms the final feature set does NOT rely on those columns, and that adding them would inflate the score artificially.")

flag_like = [c for c in df.columns if "flag" in c.lower() or "score" in c.lower() or "decision" in c.lower()]
print("\nExisting product flags/scores in raw data:", flag_like, "(none exist to leak from)")


Forbidden columns present in final feature set: set() (empty = clean)



Honest ROC-AUC (final feature set): 0.635
ROC-AUC with suspect columns added back: 0.835
Gap: +0.200 -- confirms the final feature set does NOT rely on those columns, and that adding them would inflate the score artificially.

Existing product flags/scores in raw data: [] (none exist to leak from)


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (my own real sentence, from ML-08 Section 3's printed output):** *"Both learned models beat the rule at every K here."*

**Rewritten in safe language:** In one grouped-by-client held-out test (n=7,115 rows, 8 clients unseen in training), both Logistic Regression and Random Forest were **observed** to score higher than the Week-4 rule at precision@10, @20, and @50. This is a **measured** result on a single split and a small set of held-out clients, not a general guarantee — `is_declining_label` is itself a defined threshold rule on 30-day impression change, not a confirmed real-world outcome, and 8 clients is not enough to claim the pattern generalizes. The result is best read as **directional** evidence that the learned models separate the two classes better than the hand-written rule on this data, and as **decision-support** for which approach to prefer going forward — not as proof the models will beat the rule on a different set of clients or a different time period.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the exact number the original bold claim rested on, plus how thin the sample behind it is
print(f"precision@10 on this one grouped split: {after['precision@10']:.3f}")
print(f"Test set size behind that number: {after['n_test']} rows, from {df.iloc[test_idx_grp]['client_id'].nunique()} clients")
print("A perfect top-10 on ~7K rows from 8 clients is a small, single-split result --")
print("safe language treats it as directional/decision-support, not as a proven guarantee.")


precision@10 on this one grouped split: 1.000
Test set size behind that number: 7115 rows, from 8 clients
A perfect top-10 on ~7K rows from 8 clients is a small, single-split result --
safe language treats it as directional/decision-support, not as a proven guarantee.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.